<a href="https://colab.research.google.com/github/6ocra3/LifeMatrix/blob/main/4-TextAI/HSEBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Создаём предметно-ориентированного чат-бота

Для создания чат-бота, который может отвечать на вопросы по какой-то конкретной теме или предметной области, используется подход Retrieval-Augmented Generation (RAG).

Для начала установим необходимые библиотеки. Мы будем использовать библиотеку LangChain, поскольку она содержит в себе все необходимые компоненты для создания таких чат-ботов:
* общение с языковыми моделями
* векторная база данных
* эмбеддинги

In [1]:
%pip install langchain langchain_community yandexcloud gigachat chromadb langchain_chroma huggingface_hub sentence_transformers telebot

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of grpcio-tools to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-tools to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 96.6 MB/s 

Мы будем строить бота-консультанта для Школы дизайна ВШЭ. Для построения бота нужна текстовая база знаний, содержащая всю необходимую информацию. В идеале она должны быть разбита не небольшие кусочки, каждый из которых содержит законченный фрагмент информации.

Возьмём текст с сайта ШД, немного причёсанный и разбитый на фрагменты символами `//`:

In [2]:
!wget https://github.com/shwars/ai-for-creatives/raw/refs/heads/main/data/openbac.txt

--2026-03-21 19:19:30--  https://github.com/shwars/ai-for-creatives/raw/refs/heads/main/data/openbac.txt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shwars/ai-for-creatives/refs/heads/main/data/openbac.txt [following]
--2026-03-21 19:19:31--  https://raw.githubusercontent.com/shwars/ai-for-creatives/refs/heads/main/data/openbac.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 36170 (35K) [text/plain]
Saving to: ‘openbac.txt’

openbac.txt         100%[===================>]  35.32K  --.-KB/s    in 0.003s  

2026-03-21 19:19:31 (10.7 MB/s) - ‘openbac.txt’ saved [36170/36170]



In [4]:
!head openbac.txt

ЧАСТЬ 1. ОБЩАЯ ИНФОРМАЦИЯ
Minecraft — это компьютерная игра в жанре sandbox, то есть «песочница», где игроку дают большой открытый мир и свободу действий. Основная идея игры заключается в исследовании мира, добыче ресурсов, строительстве, выживании и создании собственных проектов. Мир Minecraft состоит из блоков: земля, камень, дерево, вода, песок, руда и почти все другие объекты представлены в виде кубических элементов. Благодаря этому игрок может разрушать окружающую среду, собирать материалы и использовать их для крафта инструментов, оружия, брони и построек.

Игра не навязывает один строгий сценарий прохождения. В Minecraft можно строить дом, исследовать пещеры, выращивать еду, сражаться с монстрами, автоматизировать процессы с помощью механизмов и даже создавать целые города. Именно свобода действий сделала игру очень популярной среди игроков разного возраста.
\\
ЧАСТЬ 2. РЕЖИМЫ ИГРЫ
В Minecraft есть несколько основных режимов игры. Самый известный — выживание. В этом режиме игрок

В качестве языковой модели будем использовать Gigachat, поэтому нам потребуется ключ, который можно получить [по инструкции](https://developers.sber.ru/docs/ru/gigachat/quickstart/ind-using-api).

Импортируем необходимые библиотеки и создадим объекты:
* GPT для вызова Gigachat
* embeddings для вычисления семантических эмбеддингов текста. Для этого будем использовать мультиязычную модель [intfloat/multilingual-e5-large](https://huggingface.co/intfloat/multilingual-e5-large).

> Для вычисления эмбеддингов было бы идеально тоже использовать облачный сервис от Gigachat или Yandex, но Gigachat требует оплату за его использование.

> Модель эмбеддингов будет вычисляться внутри Colab, поэтому если не включить GPU - на индексирование всего текста может потребоваться 2-3 минуты.

In [6]:
from langchain_chroma import Chroma
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings
from langchain_community.chat_models.gigachat import GigaChat
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore')

gigachat_creds = userdata.get('gigachat_creds')

GPT = GigaChat(
    credentials=gigachat_creds,
    scope="GIGACHAT_API_PERS",
    model="GigaChat",
    streaming=False,
    verify_ssl_certs=False,
)

embeddings = HuggingFaceEmbeddings(
    model_name = "intfloat/multilingual-e5-large"
)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Загрузим документ и разобьем его на фрагменты:

In [7]:
doc = open('openbac.txt',encoding='utf-8').read()
docs = doc.split('\\')
print(f"Всего фрагментов: {len(docs)}")
print(f"Макс длина фрагмента: {max(map(len,docs))}")

Всего фрагментов: 29
Макс длина фрагмента: 942


Поскольку длина макс фрагмента не превышает 1500 символов (500 токенов), то эти фрагменты можно дополнительно не разбивать на части. С учётом такой длины мы можем добавлять в запрос 3-5 найденных фрагментов текста.

Посмотрим, как работает вычисление эмбеддингов:

In [8]:
res = embeddings.embed_documents(docs[0])
print(f"Длина эмбеддингов: {len(res[0])}")

Длина эмбеддингов: 1024


Для хранения всех документов, проиндексированных по эмбеддингам, будем использовать вектоную базу данных, например, ChromaDB. При добавлении текстов в эту базу они будут автоматически проиндексированы:

In [9]:
db = Chroma('vecstore',embeddings,'./db')
db.add_texts(docs)

['6aa9845a-8f66-4ea7-983c-73894f8639fb',
 '15743759-ac06-4d06-b79a-b09284fb867a',
 '90c36ecf-5640-46bd-8ada-3d30deca19cc',
 'a1c520de-eb2a-476d-bfdc-749d73a6945c',
 '31564e17-8298-48a4-82f8-397d307ee392',
 'e577a2b1-c6e9-482a-9097-652fdd578d05',
 '9225b304-b04a-4ab2-bb9b-a75695b8f72e',
 '5c54ba48-7c48-4d7e-a906-0774535553dd',
 '304eb3b1-8f29-4e3d-a768-a8edc7c5fc81',
 '9b80f66a-d379-4eb0-87fb-e989a866ab4e',
 '6e79afde-e979-4dd6-afdb-8b025c4f1f39',
 '51662c7d-9b07-4331-9ac3-85b2d0c9e178',
 '9034be74-25a1-4958-89fb-664a09879b1d',
 '8635f121-2d8e-4e6d-9f9c-a501976e362c',
 'ba6a3784-d544-463f-b6d6-ebfa8fa66a4c',
 'bb959d7c-58fd-432b-9bea-d040895d8be3',
 'a4ee19b0-c565-4732-9bd2-2be7ce831a8b',
 '2f5d4f15-55ad-4f4b-b2ec-d21d6fd87d27',
 '5f3557a2-4a42-4c0f-8611-f2a7ca7eb223',
 'b2dba7fb-02b5-4168-b05f-f34a3109e583',
 'ba2a6f5d-edf5-47b8-b72e-5f3a547dccaf',
 '181cda82-49da-4c13-a191-d0747516c445',
 'c3f718cd-f437-4830-83a6-dff05212a434',
 'c591d106-beff-4b0a-87c3-98a5b8c6a27e',
 'e7d6fb2f-fa5a-

Предположим, мы хотим найти среди документов ответ на вопрос **Сколько баллов ЕГЭ нужно для поступления в школу дизайна?**. Для этого определим объект `retriever`, и передадим ему параметр - количество документов, которые надо найти. После этого поиск сводится к простому вызову `retriever.invoke`

In [16]:
q = "Что ты знаешь об игре майнкрафт"

retriever = db.as_retriever(search_kwargs={"k": 3 })
retriever.invoke(q)


[Document(id='6aa9845a-8f66-4ea7-983c-73894f8639fb', metadata={}, page_content='ЧАСТЬ 1. ОБЩАЯ ИНФОРМАЦИЯ\nMinecraft — это компьютерная игра в жанре sandbox, то есть «песочница», где игроку дают большой открытый мир и свободу действий. Основная идея игры заключается в исследовании мира, добыче ресурсов, строительстве, выживании и создании собственных проектов. Мир Minecraft состоит из блоков: земля, камень, дерево, вода, песок, руда и почти все другие объекты представлены в виде кубических элементов. Благодаря этому игрок может разрушать окружающую среду, собирать материалы и использовать их для крафта инструментов, оружия, брони и построек.\n\nИгра не навязывает один строгий сценарий прохождения. В Minecraft можно строить дом, исследовать пещеры, выращивать еду, сражаться с монстрами, автоматизировать процессы с помощью механизмов и даже создавать целые города. Именно свобода действий сделала игру очень популярной среди игроков разного возраста.\n'),
 Document(id='c837a856-ffb7-4527-9

Реализуем самую главную функцию ответа на вопрос! Она работает следующим образом:

1. Вызываем `retriever` и получаем 3 релевантных фрагмента текста
2. Объединяем их вместе в контекст `context`
3. Передаём GPT-модели промпт, включающий в себя исходный вопрос и контекст. Модель в этом случае сама смотрит на контекст и находит в нём нужную информацию.

In [17]:
def answer(q):
  res = retriever.invoke(q)
  context = '\n'.join(x.page_content for x in res)
  prompt = f"""
  Прочитай следующий текст и используй его при ответе на вопрос далее.
  Используй только информацию из текста. Если в тексте нет ответа на данный вопрос,
  напиши, что не знаешь. Вот текст:\n{context}\nВопрос: {q}"""
  return GPT.invoke(prompt).content

answer(q)

'Вот основные сведения об игре Minecraft из предоставленного текста:\n\n- **Жанр**: Minecraft относится к жанру «песочницы» (sandbox), где игрокам предоставляется большая открытая среда и полная свобода действий.\n- **Основная концепция**: Игрок исследует мир, собирает ресурсы, строит постройки, защищает себя от враждебных существ и создает собственные проекты.\n- **Мир Minecraft**: Состоит из множества кубических блоков (земля, камни, деревья, вода, руды и прочие элементы).\n- **Игровая механика**: Игроки могут ломать блоки окружающей среды, добывать ресурсы, использовать их для изготовления различных инструментов, оружия, брони и сооружений.\n- **Режимы игры**:\n  - Выживание: наиболее популярный режим, подразумевающий самостоятельное добывание ресурсов, защиту от врагов и развитие персонажа.\n  - Творческий: игроки имеют неограниченные ресурсы и возможность летать, идеально подходящий для творчества и строительства.\n  - Приключенческий: режим, используемый преимущественно на пользо

А теперь оформим это всё в виде телеграм-бота! Для этого используем библиотеку `telebot`. Чтобы создать бота нам нужно сначала получить специальный токен, пообщавшись в telegram со специальным ботом [@botfather](http://t.me/botfather). Полученный токен сохраните в секретах Google Colab.

Код ниже работает в режиме поллинга - он опрашивает сервера telegram на предмет наличия сообщений, и вызывает функцию `handle_message`, если сообщение пришло. Бот в телеграме будет работать только до тех пор, пока следующая ячейка выполняется.

In [20]:
import telebot

telegram_token = userdata.get('tg_token')

bot = telebot.TeleBot(telegram_token)

# Обработчик команды /start
@bot.message_handler(commands=['start'])
def start(message):
    # Отправляем приветственное сообщение
    bot.send_message(message.chat.id,
                     'Привет, я бот, который знает всё про Майнкруфт. Спрашивай!')

# Обработчик для всех входящих сообщений
@bot.message_handler(func=lambda message: True)
def handle_message(message):
    ans = answer(message.text)
    ans += "\nЛюблю тебя котенок))"
    bot.send_message(message.chat.id, ans)

# Запуск бота
print("Бот готов к работе")
bot.polling(none_stop=True)

Бот готов к работе


### Выводы

С помощью langchain, векторной базы данных и LLM мы создали предметно-ориентированного чатбота. Качество работы такого бота будет зависеть от многих факторов:
* Насколько качественно собрана текстовая база знаний
* Насколько атомарна информация, содержащаяся во фрагментах базы знаний
* Насколько хорошо работает поиск по эмбеддингам (существуют и другие методы поиска, которые также учитывают ключевые слова)
* От качества работы LLM